# 12 — X7 Feature Pipeline (Proje Yapısına Entegre)

`data_pipeline_x7_v2.py` betiğinin mevcut proje yapısına uyarlanmış notebook versiyonu.

**Ne değişti?**
- Hardcoded `save_dir` yerine projedeki `data/raw/` ve `data/processed/` klasörleri
- Bağımsız `.npy` split yerine **`src/data/` modüllerine devirme** (split, sequence, labeling artık mevcut pipeline'da)
- Çıktı: mevcut `configs/base.yaml` ile uyumlu **CSV** + otomatik **config varyantı**
- Doğrulama: üretilen CSV mevcut `load_market_data`, `split_dev_test`, `make_expanding_folds`, `build_sequences_for_endpoints` ile test edilir

**Akış:**
```
yfinance → 5 teknik indikatör (16 ham) → Phase 1 denoising (84 türev) → 252-bar rolling z-score
         → data/raw/market_data_x7_fwdN.csv (mevcut format)
         → configs/x7_fwdN.yaml (otomatik üretilir)
         → mevcut notebook 02–11 ve run_experiment.py bu config'le çalışır
```

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    TOOLKIT_PATH = '/content/drive/MyDrive/ANN-Project/toolkit.py'
    exec(open(TOOLKIT_PATH).read())
    setup()  # repo clone/pull + deps + sys.path + cwd
    PROJECT_ROOT = Path('/content/repo')
else:
    PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import json
import subprocess

import numpy as np
import pandas as pd
import yaml

# yfinance Colab'da kurulu olmayabilir -> guvenli kontrol
try:
    import yfinance as yf
except ImportError:
    print('>> yfinance kurulu degil, yukleniyor...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance>=0.2.40'], check=True)
    import yfinance as yf

# Mevcut proje modulleri
from src.data.load_data import load_market_data, assert_chronological_order
from src.data.splitters import split_dev_test, make_expanding_folds
from src.data.sequence_builder import build_sequences_for_endpoints, drop_neutral_sequences
from src.data.labeling import compute_forward_return, compute_threshold, make_labels

print(f'IN_COLAB     : {IN_COLAB}')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'CWD          : {Path.cwd()}')

## 2. X7 Konfigürasyonu

Tüm X7 hiperparametreleri tek dict'te. `fwd_n` değerini değiştirip notebook'u tekrar çalıştırırsan farklı horizon için ayrı CSV/config üretir.

In [ ]:
X7 = {
    "ticker":        "^IXIC",
    "start":         "2010-01-01",
    "end":           "2026-04-26",
    "fwd_n":         5,        # PHASE 2: 1 / 5 / 21 ile dene
    # İndikatör pencereleri
    "vrvp_window":   252,
    "nw_window":     100,
    "nw_h":          25,
    "zscore_window": 252,
    # Phase 1 denoising
    "denoise_short": 5,
    "denoise_long":  21,
}
X7["gap"] = max(52, X7["nw_window"], X7["fwd_n"])

# Proje yapısına uygun çıktı yolları
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_DIR    = PROJECT_ROOT / "configs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TAG          = f"x7_fwd{X7['fwd_n']}"
CSV_PATH     = RAW_DIR       / f"market_data_{TAG}.csv"
META_PATH    = PROCESSED_DIR / f"{TAG}_meta.json"
CONFIG_PATH  = CONFIG_DIR    / f"{TAG}.yaml"

print(f"CSV     → {CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"Meta    → {META_PATH.relative_to(PROJECT_ROOT)}")
print(f"Config  → {CONFIG_PATH.relative_to(PROJECT_ROOT)}")

## 3. Veri Çekme

yfinance üzerinden NASDAQ OHLCV. Mevcut `assert_chronological_order` ile sıra doğrulanıyor.

In [ ]:
print(f"Fetching {X7['ticker']} from {X7['start']} to {X7['end']}...")
df = yf.download(X7["ticker"], start=X7["start"], end=X7["end"], progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
df.index.name = "Date"

# Mevcut proje doğrulamasıyla kontrol
assert_chronological_order(df.reset_index(), date_col="Date")

print(f"Loaded {len(df)} bars  |  {df.index[0].date()} → {df.index[-1].date()}")
df.head()

## 4. Ham Feature'lar — 5 Teknik İndikatör (16 sütun)

Tümü point-in-time: sadece `rolling`/`shift`/`ewm` — leakage yok.

### 4.1 VRVP (Volume Relative Value Profile, 252-bar)

In [ ]:
print("Computing VRVP...")
W       = X7["vrvp_window"]
lows    = df["Low"].values
highs   = df["High"].values
closes  = df["Close"].values
volumes = df["Volume"].values
prices  = (highs + lows) / 2.0

poc = np.full(len(df), np.nan)
vah = np.full(len(df), np.nan)
val = np.full(len(df), np.nan)

for i in range(W - 1, len(df)):
    s = i - W + 1
    w_p = prices[s:i+1]
    w_v = volumes[s:i+1]
    w_lo = np.min(lows[s:i+1])
    w_hi = np.max(highs[s:i+1])

    if w_hi == w_lo:
        poc[i] = vah[i] = val[i] = w_p[-1]
        continue

    bins = np.linspace(w_lo, w_hi, 21)
    dig  = np.clip(np.digitize(w_p, bins), 1, 20)
    profile = np.array([w_v[dig == b].sum() for b in range(1, 21)])

    p_idx = int(np.argmax(profile))
    poc_v = (bins[p_idx] + bins[p_idx + 1]) / 2.0
    target = w_v.sum() * 0.70
    cur = profile[p_idx]
    lo, hi = p_idx, p_idx

    while cur < target and (lo > 0 or hi < 19):
        L = profile[lo - 1] if lo > 0 else 0
        R = profile[hi + 1] if hi < 19 else 0
        if L > R:
            lo -= 1; cur += L
        elif R > L:
            hi += 1; cur += R
        else:
            if L == 0 and R == 0:
                break
            lo -= 1; hi += 1; cur += L + R

    vah[i] = bins[min(19, hi) + 1]
    val[i] = bins[max(0, lo)]
    poc[i] = poc_v

df["vrvp_poc_dist"] = (poc - df["Close"]) / df["Close"]
df["vrvp_vah_dist"] = (vah - df["Close"]) / df["Close"]
df["vrvp_val_dist"] = (val - df["Close"]) / df["Close"]

raw_features = ["vrvp_poc_dist", "vrvp_vah_dist", "vrvp_val_dist"]
print(f"  + {len(raw_features)} VRVP features")

### 4.2 Ichimoku Cloud (lagged senkou)

In [ ]:
print("Computing Ichimoku...")
tenkan   = (df["High"].rolling(9).max()  + df["Low"].rolling(9).min())  / 2.0
kijun    = (df["High"].rolling(26).max() + df["Low"].rolling(26).min()) / 2.0
senkou_a = ((tenkan + kijun) / 2.0).shift(26)
senkou_b = ((df["High"].rolling(52).max() + df["Low"].rolling(52).min()) / 2.0).shift(26)

df["ichi_sa_dist"]     = (df["Close"] - senkou_a) / df["Close"]
df["ichi_sb_dist"]     = (df["Close"] - senkou_b) / df["Close"]
df["ichi_tk_cross"]    = np.sign(tenkan - kijun)
df["ichi_cloud_thick"] = np.abs(senkou_a - senkou_b) / df["Close"]

raw_features += ["ichi_sa_dist", "ichi_sb_dist", "ichi_tk_cross", "ichi_cloud_thick"]
print("  + 4 Ichimoku features")

### 4.3 Nadaraya-Watson Envelope (Gaussian kernel)

In [ ]:
print("Computing Nadaraya-Watson...")
W, h = X7["nw_window"], X7["nw_h"]
j = np.arange(W)
kern = np.exp(-0.5 * ((j[:, None] - j[None, :]) / h) ** 2)
ksum = kern.sum(axis=1)

closes_arr = df["Close"].values
nw_mid = np.full(len(df), np.nan)
nw_up  = np.full(len(df), np.nan)
nw_lo  = np.full(len(df), np.nan)

for i in range(W - 1, len(df)):
    window = closes_arr[i - W + 1 : i + 1]
    curve  = (kern @ window) / ksum
    nw_mid[i] = curve[-1]
    std       = np.std(window - curve)
    nw_up[i]  = curve[-1] + 2 * std
    nw_lo[i]  = curve[-1] - 2 * std

df["nw_mid_dist"]   = (df["Close"] - nw_mid) / df["Close"]
df["nw_upper_dist"] = (df["Close"] - nw_up)  / df["Close"]
df["nw_lower_dist"] = (df["Close"] - nw_lo)  / df["Close"]

raw_features += ["nw_mid_dist", "nw_upper_dist", "nw_lower_dist"]
print("  + 3 NW features")

### 4.4 KAMA (Kaufman Adaptive Moving Average)

In [ ]:
print("Computing KAMA...")
change     = np.abs(df["Close"] - df["Close"].shift(10))
volatility = np.abs(df["Close"].diff()).rolling(10).sum()
er         = change / volatility

fast_sc = 2.0 / 3.0
slow_sc = 2.0 / 31.0
sc      = (er * (fast_sc - slow_sc) + slow_sc) ** 2

kama    = np.full(len(df), np.nan)
sc_vals = sc.values
if len(df) > 10:
    kama[10] = closes_arr[10]
    for i in range(11, len(df)):
        if not np.isnan(sc_vals[i]):
            kama[i] = kama[i - 1] + sc_vals[i] * (closes_arr[i] - kama[i - 1])
        else:
            kama[i] = kama[i - 1]

df["kama_dist"] = (df["Close"] - kama) / df["Close"]
df["kama_er"]   = er

raw_features += ["kama_dist", "kama_er"]
print("  + 2 KAMA features")

### 4.5 Supertrend (ATR-based)

In [ ]:
print("Computing Supertrend...")
hl = df["High"] - df["Low"]
hc = np.abs(df["High"] - df["Close"].shift(1))
lc = np.abs(df["Low"]  - df["Close"].shift(1))
tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
atr = tr.ewm(alpha=1/14, adjust=False).mean()

hl2 = (df["High"] + df["Low"]) / 2.0
bu  = (hl2 + 2.0 * atr).values
bl  = (hl2 - 2.0 * atr).values

fu = np.zeros(len(df))
fl = np.zeros(len(df))
st = np.zeros(len(df))
fu[0], fl[0], st[0] = bu[0], bl[0], 1

for i in range(1, len(df)):
    fu[i] = bu[i] if (bu[i] < fu[i-1] or closes_arr[i-1] > fu[i-1]) else fu[i-1]
    fl[i] = bl[i] if (bl[i] > fl[i-1] or closes_arr[i-1] < fl[i-1]) else fl[i-1]
    if   st[i-1] ==  1 and closes_arr[i] <= fu[i]: st[i] = -1
    elif st[i-1] == -1 and closes_arr[i] >= fl[i]: st[i] =  1
    else: st[i] = st[i-1]

df["st_direction"]  = st
df["st_lower_dist"] = (df["Close"] - fl) / df["Close"]
df["st_upper_dist"] = (fu - df["Close"]) / df["Close"]
df["st_atr_pct"]    = atr / df["Close"]

raw_features += ["st_direction", "st_lower_dist", "st_upper_dist", "st_atr_pct"]
print(f"  + 4 Supertrend features")
print(f"\nToplam ham feature: {len(raw_features)}")

## 5. Phase 1 — Feature Denoising (84 türev)

Binary feature'lar (`st_direction`, `ichi_tk_cross`) atlanır → 14 sürekli × 6 türev = **84 türev sütun**.

| Türev | Formül | Anlam |
|---|---|---|
| `__delta_5` | `series.diff(5)` | 5 günlük kısa momentum |
| `__delta_21` | `series.diff(21)` | 21 günlük orta vade momentum |
| `__ewm_5` | EWM span=5 | Hızlı smooth trend |
| `__ewm_21` | EWM span=21 | Yavaş smooth trend |
| `__residual_21` | `series − ewm_21` | Yüksek frekanslı gürültü |
| `__rolling_z_21` | 21-bar z-score, clip(±5) | Anlık normalize sapma |

In [ ]:
SKIP_BINARY = {"st_direction", "ichi_tk_cross"}
s, lg = X7["denoise_short"], X7["denoise_long"]

derived_features = []
for col in raw_features:
    if col in SKIP_BINARY:
        continue
    series = df[col]

    df[f"{col}__delta_{s}"]  = series.diff(s)
    df[f"{col}__delta_{lg}"] = series.diff(lg)
    df[f"{col}__ewm_{s}"]    = series.ewm(span=s,  adjust=False).mean()
    df[f"{col}__ewm_{lg}"]   = series.ewm(span=lg, adjust=False).mean()
    df[f"{col}__residual_{lg}"] = series - df[f"{col}__ewm_{lg}"]

    rmean = series.rolling(lg).mean()
    rstd  = series.rolling(lg).std()
    df[f"{col}__rolling_z_{lg}"] = ((series - rmean) / (rstd + 1e-8)).clip(-5, 5)

    derived_features += [
        f"{col}__delta_{s}",  f"{col}__delta_{lg}",
        f"{col}__ewm_{s}",    f"{col}__ewm_{lg}",
        f"{col}__residual_{lg}", f"{col}__rolling_z_{lg}",
    ]

print(f"Sürekli ham : {len(raw_features) - len(SKIP_BINARY)}")
print(f"Türev sayısı: {len(derived_features)}  ({(len(raw_features) - len(SKIP_BINARY))} × 6)")
print(f"TOPLAM      : {len(raw_features) + len(derived_features)} feature")

## 6. Rolling Z-score Normalizasyonu (252-bar)

Binary feature'lar ve zaten z-score olan türevler atlanır. Sadece geçmişe bakan rolling pencere → leakage yok.

In [ ]:
ZW = X7["zscore_window"]
SKIP_EXACT  = {"st_direction", "ichi_tk_cross"}
SKIP_SUFFIX = ("__rolling_z_",)

all_features = raw_features + derived_features
n_normalized = 0

for col in all_features:
    base = col.split("__")[0]
    if base in SKIP_EXACT:
        continue
    if any(col.endswith(suf) for suf in SKIP_SUFFIX):
        continue
    m = df[col].rolling(ZW).mean()
    sd = df[col].rolling(ZW).std()
    df[col] = ((df[col] - m) / (sd + 1e-8)).clip(-5, 5)
    n_normalized += 1

print(f"Normalized {n_normalized} / {len(all_features)} features (252-bar rolling z-score).")

## 7. Mevcut Pipeline Formatında CSV Çıktısı

CSV kolonları: `Date | Nasdaq_Close | <100 feature>` — bu yapı `configs/base.yaml`'in beklediği şema.
Warm-up NaN satırları silinir (en az 252-bar rolling pencere gerekiyordu).

In [ ]:
# Mevcut config'le uyumlu CSV: Date | Nasdaq_Close | features...
output_df = df[["Close"] + all_features].copy()
output_df = output_df.rename(columns={"Close": "Nasdaq_Close"})
output_df.index.name = "Date"

# Warm-up NaN'leri (en az 252-bar pencere)
before = len(output_df)
output_df = output_df.dropna()
after = len(output_df)

output_df.to_csv(CSV_PATH)

print(f"Dropped {before - after} warm-up rows  →  {after} usable bars")
print(f"Date range  : {output_df.index[0].date()} → {output_df.index[-1].date()}")
print(f"Shape       : {output_df.shape}")
print(f"Saved CSV   : {CSV_PATH}")
output_df.head()

## 8. Otomatik Config Varyantı Üretimi

Mevcut `base.yaml` şablon olarak alınır; sadece `data.*` ve `labeling.horizon` override edilir. Üretilen `configs/x7_fwdN.yaml` mevcut tüm pipeline'larla **tak-çalıştır** uyumludur.

In [ ]:
base_config_path = CONFIG_DIR / "base.yaml"
with open(base_config_path, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

base_cfg["experiment"]["name"]  = TAG
base_cfg["experiment"]["notes"] = f"X7 100-feature pipeline (fwd_n={X7['fwd_n']})"

base_cfg["data"]["file_path"]   = str(CSV_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/")
base_cfg["data"]["date_col"]    = "Date"
base_cfg["data"]["close_col"]   = "Nasdaq_Close"
base_cfg["data"]["feature_cols"] = all_features

# X7'nin forward horizon'unu mevcut labeling.horizon'a yansıt
base_cfg["labeling"]["horizon"] = X7["fwd_n"]

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(base_cfg, f, sort_keys=False, default_flow_style=False, allow_unicode=True)

print(f"Saved config: {CONFIG_PATH}")
print(f"  experiment.name  : {base_cfg['experiment']['name']}")
print(f"  data.file_path   : {base_cfg['data']['file_path']}")
print(f"  data.close_col   : {base_cfg['data']['close_col']}")
print(f"  feature_cols     : {len(base_cfg['data']['feature_cols'])} features")
print(f"  labeling.horizon : {base_cfg['labeling']['horizon']}")

## 9. Mevcut Modüllerle End-to-End Doğrulama

Üretilen CSV'nin projedeki pipeline ile **gerçekten** çalıştığını gösteren smoke test. Aşağıdaki adımlar `run_experiment.py`'ın aynısını yapar — sadece daha küçük ölçekte.

In [ ]:
# 1) CSV yukleme (mevcut loader)
df_loaded = load_market_data(str(CSV_PATH), date_col='Date')
print(f"1. load_market_data         OK  shape={df_loaded.shape}")

# 2) Dev/Test split (mevcut)
dev_df, test_df = split_dev_test(df_loaded, test_ratio=0.15)
print(f"2. split_dev_test           OK  dev={len(dev_df)}, test={len(test_df)}")

# 3) Expanding CV folds (mevcut) - folds = List[Tuple[range, range]]
folds = make_expanding_folds(
    n_rows=len(dev_df),
    n_folds=4,
    val_ratio_within_dev=0.10,
)
print(f"3. make_expanding_folds     OK  {len(folds)} folds")
for i, (tr, va) in enumerate(folds, start=1):
    print(f"     Fold {i}: train[{tr.start}:{tr.stop}]  val[{va.start}:{va.stop}]")

# 4) Forward return + threshold + labeling (mevcut)
fwd_ret   = compute_forward_return(dev_df['Nasdaq_Close'], horizon=X7['fwd_n'])
threshold = compute_threshold(fwd_ret.dropna(), method='quantile', quantile=0.40)
labels    = make_labels(fwd_ret, threshold=threshold)
label_counts = labels.value_counts().to_dict()
print(f"4. labeling (h={X7['fwd_n']}, q=0.40)  OK  thresh={threshold:.5f}  dist={label_counts}")

# 5) Sequence builder (mevcut) - fold 1 train kisminda ornek
tr_range = folds[0][0]   # range objesi
lookback = 21
endpoint_idx = list(range(max(lookback - 1, tr_range.start), tr_range.stop))
print(f"   Endpoint count: {len(endpoint_idx)} (train range: {tr_range.start}..{tr_range.stop})")

X_seq, y_seq, ts, ep = build_sequences_for_endpoints(
    df=dev_df,
    feature_cols=all_features,
    labels=labels,
    endpoint_indices=endpoint_idx,
    lookback=lookback,
)
X_seq, y_seq, ts = drop_neutral_sequences(X_seq, y_seq, ts, neutral_value=-1)
print(f"5. build_sequences          OK  X={X_seq.shape}  y={y_seq.shape}  (after dropping neutrals)")

print()
print("=" * 60)
print("  OK  X7 verisi mevcut pipeline ile tam uyumlu calisiyor")
print("=" * 60)

## 10. Metadata Kaydı

Pipeline çıktıları + hiperparametreler tek JSON'da. Projedeki `artifacts/` formatına benzer.

In [ ]:
meta = {
    "tag": TAG,
    "ticker": X7["ticker"],
    "date_range": [str(df.index[0].date()), str(df.index[-1].date())],
    "n_bars_fetched": int(len(df)),
    "n_bars_after_warmup": int(len(output_df)),
    "fwd_n": X7["fwd_n"],
    "n_features_raw": len(raw_features),
    "n_features_derived": len(derived_features),
    "n_features_total": len(all_features),
    "raw_features": raw_features,
    "derived_features": derived_features,
    "hyperparameters": {
        "vrvp_window":   X7["vrvp_window"],
        "nw_window":     X7["nw_window"],
        "nw_h":          X7["nw_h"],
        "zscore_window": X7["zscore_window"],
        "denoise_short": X7["denoise_short"],
        "denoise_long":  X7["denoise_long"],
        "gap":           X7["gap"],
    },
    "outputs": {
        "csv":    str(CSV_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "config": str(CONFIG_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "meta":   str(META_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/"),
    },
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata: {META_PATH}")
print(json.dumps(meta["outputs"], indent=2))

## 11. Colab — Drive'a Kalıcı Kayıt

`/content/repo` ephemeral'dir; session bitince silinir. Bu hücre Colab'da çıktıları Drive'a kopyalar:

1. **`drive/MyDrive/ANN-Project/data/raw/`** → toolkit'in `_link_data()` fonksiyonu sonraki session'larda bunu otomatik repo içine kopyalar (toolkit.py:68-79). Yani notebook 09 vb. açıldığında X7 CSV hazır olur.
2. **`drive/MyDrive/ANN-Project/configs/`** → config yaml de kalıcılaşır.
3. **`drive/MyDrive/ANN-Project/x7_outputs/`** → meta + yedek (referans için).

Lokal çalıştırmada bu hücre no-op.

In [ ]:
import shutil

if IN_COLAB:
    DRIVE_ROOT     = Path('/content/drive/MyDrive/ANN-Project')
    DRIVE_RAW      = DRIVE_ROOT / 'data' / 'raw'
    DRIVE_CONFIGS  = DRIVE_ROOT / 'configs'
    DRIVE_X7       = DRIVE_ROOT / 'x7_outputs'

    for d in (DRIVE_RAW, DRIVE_CONFIGS, DRIVE_X7):
        d.mkdir(parents=True, exist_ok=True)

    # 1) CSV -> Drive/data/raw/  (toolkit _link_data() bunu sonraki session'larda repoya kopyalar)
    csv_dst = DRIVE_RAW / CSV_PATH.name
    shutil.copy(CSV_PATH, csv_dst)
    print(f'  -> {csv_dst}')

    # 2) Config -> Drive/configs/
    cfg_dst = DRIVE_CONFIGS / CONFIG_PATH.name
    shutil.copy(CONFIG_PATH, cfg_dst)
    print(f'  -> {cfg_dst}')

    # 3) Meta + yedek -> Drive/x7_outputs/
    meta_dst = DRIVE_X7 / META_PATH.name
    shutil.copy(META_PATH, meta_dst)
    print(f'  -> {meta_dst}')

    # Yedek olarak CSV ve config de buraya
    shutil.copy(CSV_PATH,    DRIVE_X7 / CSV_PATH.name)
    shutil.copy(CONFIG_PATH, DRIVE_X7 / CONFIG_PATH.name)

    print()
    print('Drive persistence OK. Sonraki Colab session aciliminda:')
    print('  - toolkit.setup() CSV i otomatik /content/repo/data/raw/ icine kopyalar')
    print('  - Config dosyasini manuel kopyalaman gerekir:')
    print(f'      !cp {cfg_dst} /content/repo/configs/')
else:
    print('Lokal calistirma - Drive kopyalama atlandi.')
    print(f'Dosyalar zaten projede:')
    print(f'  CSV    : {CSV_PATH}')
    print(f'  Config : {CONFIG_PATH}')
    print(f'  Meta   : {META_PATH}')

## 12. Sonraki Adımlar — X7 Feature Seti ile Eğitim

X7 feature seti artık projedeki tüm akışla çalışır.

### Komut satırından tam pipeline
```bash
python src/run_experiment.py --config configs/x7_fwd5.yaml
```

### Python'dan
```python
from src.run_experiment import main
main(config_path="configs/x7_fwd5.yaml")
```

### Mevcut notebook'larda kullanmak
Notebook 02-11'in başındaki `CONFIG_PATH`'i değiştir:
```python
CONFIG_PATH = ROOT / 'configs' / 'x7_fwd5.yaml'   # base.yaml yerine
```

### Farklı horizon denemek
Notebook'un başındaki `X7["fwd_n"]` değerini `1`, `5` veya `21` yap, notebook'u tekrar çalıştır. Her biri ayrı CSV ve config üretir:
- `configs/x7_fwd1.yaml`  →  yarın yönü
- `configs/x7_fwd5.yaml`  →  1 hafta yönü  (önerilen)
- `configs/x7_fwd21.yaml` →  1 ay yönü

### Beklenen kazanım
Mevcut 7-feature setine göre **100 X7 feature** + fwd_n=5 horizon, [CHANGELOG.md](../CHANGELOG.md)'de tespit edilen iki bottleneck'i adresler:
1. **Feature-label time scale mismatch** (fwd_n=1 → 5/21)
2. **Düşük feature çeşitliliği** (7 makro → 16 ham + 84 türev teknik)